# Siamese BiLSTM — Contrastive Retrieval

Neculoiu et al. (2016) Siamese Recurrent Network for learning text similarity.
Uses word2vec embedding, 4 BiLSTM layers, contrastive loss, Adam optimizer.
Trains on (question, answer) pairs with 4:1 negative ratio.

In [ ]:
!git clone https://github.com/hyperformancelabs/vnlegal-rag.git
%cd vnlegal-rag

In [ ]:
!git fetch origin
!git reset --hard origin/$(git branch --show-current)
!git clean -fd

In [ ]:
%cd vnlegal-rag

In [ ]:
import json, random, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from src.tokenizer import simple_tokenize, build_vocab, build_embedding_matrix, PAD_TOKEN, UNK_TOKEN

In [ ]:
# Device and seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Data and artifact paths
DATA_DIR = Path('data/data_ready')
ARTIFACT_DIR = Path('experiments/siamese_bilstm_artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Word2vec path — merge parts first: !cat data/word2vec/word2vec_part_* > data/word2vec/merged.zip
# Then: !7z x data/word2vec/merged.zip -odata/word2vec/
W2V_PATH = Path('data/word2vec/word2vec_vi_syllables_300dims.txt')

print('DATA_DIR:', DATA_DIR)
print('W2V_PATH exists:', W2V_PATH.exists())

In [ ]:
# Load QA splits (needs question, answer columns)
qa_train = pd.read_csv(DATA_DIR / 'qa_train.csv', sep='	')
qa_val   = pd.read_csv(DATA_DIR / 'qa_val.csv',   sep='	')
qa_test  = pd.read_csv(DATA_DIR / 'qa_test.csv',  sep='	')

# Also load corpus for full retrieval eval
corpus_full = pd.read_csv(DATA_DIR / 'corpus_full.csv', sep='	')
corpus_test = pd.read_csv(DATA_DIR / 'corpus_test.csv', sep='	')

required = {'question', 'answer'}
for name, df in [('train', qa_train), ('val', qa_val), ('test', qa_test)]:
    miss = required - set(df.columns)
    if miss:
        raise ValueError(f'{name} missing: {miss}')

print('train:', qa_train.shape, 'val:', qa_val.shape, 'test:', qa_test.shape)
print('corpus full:', corpus_full.shape, 'corpus test:', corpus_test.shape)

In [ ]:
# ── Siamese BiLSTM hyperparams (Neculoiu et al. 2016) ──
MAX_LEN = 256      # Word-level; questions p95=90, answers p95=370, balance

# Architecture
NUM_BILSTM_LAYERS = 4
HIDDEN_DIM  = 64       # Per direction; output = 128-d after concat
DENSE_DIM   = 128      # Final projection after mean-pool

# Regularization
DROPOUT_RECURRENT = 0.2   # On LSTM recurrent connections
DROPOUT_INTER     = 0.4   # Between LSTM layers
DROPOUT_OUT       = 0.4   # Before final dense projection

# Contrastive loss
CONTRASTIVE_MARGIN  = 0.5   # margin m: negative pairs with cos > m get penalized
NEGATIVE_RATIO      = 4     # negative:positive sampling ratio
POSITIVE_LOSS_SCALE = 0.25  # L+ = scale * (1 - cos)^2  (paper §3.1)

# Training
BATCH_SIZE = 64         # Pair count per batch
ADAM_LR    = 0.001      # Adam learning rate (paper uses default Adam)
GRAD_CLIP  = 5.0        # Global gradient clip
EPOCHS     = 30
PATIENCE   = 5          # Early stopping on val loss

In [ ]:
# Build vocab from QA pairs (questions + answers)
all_texts = pd.concat([
    qa_train['question'], qa_val['question'], qa_test['question'],
    qa_train['answer'],   qa_val['answer'],   qa_test['answer'],
], ignore_index=True).tolist()

stoi = build_vocab(all_texts, max_vocab=100_000, min_freq=2)
pad_idx = stoi[PAD_TOKEN]
itos = {i: w for w, i in stoi.items()}
print(f'Vocab size: {len(stoi)}')

# Build pretrained embedding matrix
embedding_weight, hits = build_embedding_matrix(stoi, embed_dim=300, w2v_path=W2V_PATH)
print(f'Embedding shape: {embedding_weight.shape} | w2v hits: {hits}/{len(stoi)}')

In [ ]:
# Tokenize all texts
def encode_texts(texts, max_len):
    ids = np.full((len(texts), max_len), pad_idx, dtype=np.int64)
    for i, t in enumerate(texts):
        tokens = simple_tokenize(str(t))[:max_len]
        ids[i, :len(tokens)] = [stoi.get(tok, stoi[UNK_TOKEN]) for tok in tokens]
    return ids

train_q_ids = encode_texts(qa_train['question'].tolist(), MAX_LEN)
train_a_ids = encode_texts(qa_train['answer'].tolist(), MAX_LEN)
val_q_ids   = encode_texts(qa_val['question'].tolist(), MAX_LEN)
val_a_ids   = encode_texts(qa_val['answer'].tolist(), MAX_LEN)
test_q_ids  = encode_texts(qa_test['question'].tolist(), MAX_LEN)
test_a_ids  = encode_texts(qa_test['answer'].tolist(), MAX_LEN)

# Corpus passages for retrieval eval
corpus_ids  = encode_texts(corpus_full['article_content'].tolist(), MAX_LEN)
corpus_test_ids = encode_texts(corpus_test['article_content'].tolist(), MAX_LEN)

q_lens = [len(simple_tokenize(str(t))) for t in qa_train['question']]
a_lens = [len(simple_tokenize(str(t))) for t in qa_train['answer']]
print(f'Question: max={max(q_lens)} p95={np.percentile(q_lens, 95):.0f}')
print(f'Answer:   max={max(a_lens)} p95={np.percentile(a_lens, 95):.0f}')
pct_q = 100 * sum(1 for L in q_lens if L > MAX_LEN) / max(len(q_lens), 1)
pct_a = 100 * sum(1 for L in a_lens if L > MAX_LEN) / max(len(a_lens), 1)
print(f'Truncated at {MAX_LEN}: questions {pct_q:.1f}%  answers {pct_a:.1f}%')

## Siamese BiLSTM Architecture

- **4 BiLSTM layers**, hidden=64 per direction → 128-d output
- **Mean-pool** over time → fixed-size vector
- **Dense projection** to final embedding
- **Cosine similarity** between query and passage embeddings
- **Contrastive loss**: L+ = 0.25×(1-cos)², L- = cos² if cos > margin

In [ ]:
class BiLSTMEncoder(nn.Module):
    """4 BiLSTM layers → mean-pool → dense → L2-normalized embedding."""

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 300,
        hidden_dim: int = 64,
        num_layers: int = 4,
        dense_dim: int = 128,
        dropout_recurrent: float = 0.2,
        dropout_inter: float = 0.4,
        dropout_out: float = 0.4,
        pad_idx: int = 0,
        embedding_weight: torch.Tensor | None = None,
        freeze_embedding: bool = False,
    ):
        super().__init__()
        if embedding_weight is not None:
            self.embedding = nn.Embedding.from_pretrained(
                embedding_weight, freeze=freeze_embedding, padding_idx=pad_idx,
            )
        else:
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers, batch_first=True, bidirectional=True,
            dropout=dropout_recurrent if num_layers > 1 else 0.0,
        )
        self.inter_dropout = nn.Dropout(dropout_inter)
        self.out_dropout = nn.Dropout(dropout_out)
        self.dense = nn.Linear(hidden_dim * 2, dense_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len)
        mask = (x != pad_idx).float()
        emb = self.embedding(x)                            # (B, T, E)
        out, _ = self.lstm(emb)                            # (B, T, 2*H)
        out = self.inter_dropout(out)
        # Masked mean-pool
        mask = mask.unsqueeze(-1)
        summed = (out * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        pooled = summed / counts                          # (B, 2*H)
        projected = self.dense(self.out_dropout(pooled))   # (B, D)
        return F.normalize(projected, p=2, dim=1)


class SiameseBiLSTM(nn.Module):
    """Siamese network: same encoder for query and passage, cosine similarity."""

    def __init__(self, **encoder_kwargs):
        super().__init__()
        self.encoder = BiLSTMEncoder(**encoder_kwargs)

    def forward(self, q: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        # returns cosine similarity for each pair
        q_emb = self.encoder(q)
        a_emb = self.encoder(a)
        return (q_emb * a_emb).sum(dim=1)


class ContrastiveLoss(nn.Module):
    """Paper §3.1: L+ = scale*(1-cos)^2, L- = cos^2 (if cos > margin)."""

    def __init__(self, margin: float = 0.5, positive_scale: float = 0.25):
        super().__init__()
        self.margin = margin
        self.positive_scale = positive_scale

    def forward(self, cos_sim: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        pos_mask = labels == 1
        neg_mask = labels == 0
        loss_pos = self.positive_scale * (1 - cos_sim[pos_mask]) ** 2
        cos_neg = cos_sim[neg_mask]
        loss_neg = cos_neg[torch.where(cos_neg < self.margin)] ** 2
        total_samples = pos_mask.sum() + max(neg_mask.sum(), 1)
        return (loss_pos.sum() + loss_neg.sum()) / total_samples


# Instantiate model
model = SiameseBiLSTM(
    vocab_size=len(stoi),
    embed_dim=300,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_BILSTM_LAYERS,
    dense_dim=DENSE_DIM,
    dropout_recurrent=DROPOUT_RECURRENT,
    dropout_inter=DROPOUT_INTER,
    dropout_out=DROPOUT_OUT,
    pad_idx=pad_idx,
    embedding_weight=embedding_weight,
    freeze_embedding=False,
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params: {total_params:,} total | {trainable_params:,} trainable')

criterion = ContrastiveLoss(margin=CONTRASTIVE_MARGIN, positive_scale=POSITIVE_LOSS_SCALE)
optimizer = torch.optim.Adam(model.parameters(), lr=ADAM_LR)

In [ ]:
class SiamesePairDataset(Dataset):
    """On-the-fly positive/negative pair generation at 1:NEGATIVE_RATIO."""

    def __init__(self, q_ids: np.ndarray, a_ids: np.ndarray, neg_ratio: int = 4):
        self.q_ids = q_ids
        self.a_ids = a_ids
        self.n = len(q_ids)
        self.neg_ratio = neg_ratio
        # Total pairs per epoch = n * (1 + neg_ratio)
        self.total_pairs = self.n * (1 + neg_ratio)

    def __len__(self):
        return self.total_pairs

    def __getitem__(self, idx):
        if idx < self.n:
            # Positive pair
            return (
                torch.tensor(self.q_ids[idx], dtype=torch.long),
                torch.tensor(self.a_ids[idx], dtype=torch.long),
                torch.tensor(1, dtype=torch.float32),
            )
        else:
            # Negative pair — random wrong answer
            i = random.randrange(self.n)
            j = random.randrange(self.n)
            while j == i:
                j = random.randrange(self.n)
            return (
                torch.tensor(self.q_ids[i], dtype=torch.long),
                torch.tensor(self.a_ids[j], dtype=torch.long),
                torch.tensor(0, dtype=torch.float32),
            )


train_ds = SiamesePairDataset(train_q_ids, train_a_ids, neg_ratio=NEGATIVE_RATIO)
val_ds   = SiamesePairDataset(val_q_ids,   val_a_ids,   neg_ratio=NEGATIVE_RATIO)
test_ds  = SiamesePairDataset(test_q_ids,  test_a_ids,  neg_ratio=NEGATIVE_RATIO)

nw = 0 if sys.platform == 'win32' else 2
pin = torch.cuda.is_available()

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=nw, pin_memory=pin)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=nw, pin_memory=pin)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=nw, pin_memory=pin)

print(f'Train pairs: {len(train_ds):,} ({train_ds.n:,} pos + {train_ds.n * NEGATIVE_RATIO:,} neg)')
print(f'Val   pairs: {len(val_ds):,}')
print(f'Test  pairs: {len(test_ds):,}')

In [ ]:
import sys
from torch.cuda.amp import GradScaler, autocast

scaler = GradScaler('cuda', enabled=torch.cuda.is_available())


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for q, a, y in loader:
        q, a, y = q.to(device), a.to(device), y.to(device)
        optimizer.zero_grad()
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            cos_sim = model(q, a)
            loss = criterion(cos_sim, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * q.size(0)
    return total_loss / len(loader.dataset)


def evaluate_pairs(model, loader):
    """Evaluate contrastive loss + binary accuracy on pairs."""
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    with torch.no_grad():
        for q, a, y in loader:
            q, a, y = q.to(device), a.to(device), y.to(device)
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                cos_sim = model(q, a)
                loss = criterion(cos_sim, y)
            total_loss += loss.item() * q.size(0)
            pred = (cos_sim >= 0.5).float()
            y_true.extend(y.cpu().tolist())
            y_pred.extend(pred.cpu().tolist())
    acc = sum(1 for t, p in zip(y_true, y_pred) if t == p) / max(len(y_true), 1)
    pos_acc = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1) / max(sum(y_true), 1)
    neg_acc = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0) / max(len(y_true) - sum(y_true), 1)
    return {
        'loss': total_loss / len(loader.dataset),
        'acc': acc, 'pos_acc': pos_acc, 'neg_acc': neg_acc,
        'y_true': y_true, 'y_pred': y_pred,
    }

In [ ]:
def retrieval_eval(model, q_ids, a_ids, corpus_ids, top_k=10):
    """Full retrieval: rank corpus passages by cosine, measure mAP and recall@k."""
    model.eval()
    batch_size = 256
    with torch.no_grad():
        # Encode queries
        q_embs = []
        for i in range(0, len(q_ids), batch_size):
            b = torch.tensor(q_ids[i:i+batch_size], dtype=torch.long, device=device)
            q_embs.append(model.encoder(b).cpu())
        q_embs = torch.cat(q_embs, dim=0)

        # Encode corpus
        c_embs = []
        for i in range(0, len(corpus_ids), batch_size):
            b = torch.tensor(corpus_ids[i:i+batch_size], dtype=torch.long, device=device)
            c_embs.append(model.encoder(b).cpu())
        c_embs = torch.cat(c_embs, dim=0)

    # Cosine similarity matrix: (n_queries, n_corpus)
    sim = F.normalize(q_embs, p=2, dim=1) @ F.normalize(c_embs, p=2, dim=1).T

    # For each query, find where true answer is in ranking
    n_queries = len(q_ids)
    # We use answer embeddings as proxy for correct passage
    a_embs = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(a_ids), batch_size):
            b = torch.tensor(a_ids[i:i+batch_size], dtype=torch.long, device=device)
            a_embs.append(model.encoder(b).cpu())
        a_embs = torch.cat(a_embs, dim=0)

    # Nearest corpus passage to each answer → ground truth idx
    a_sim = F.normalize(a_embs, p=2, dim=1) @ F.normalize(c_embs, p=2, dim=1).T
    gt_idx = a_sim.argmax(dim=1)  # (n_queries,)

    # Ranking metrics
    ranks = []
    for i in range(n_queries):
        target = gt_idx[i].item()
        rank = (sim[i] > sim[i, target]).sum().item() + 1
        ranks.append(rank)

    ranks = np.array(ranks)
    mrr = (1.0 / ranks).mean()
    recall_at_k = {k: (ranks <= k).mean() for k in [1, 5, 10, 20]}
    map_score = sum(1.0 / r for r in ranks) / n_queries  # mAP when 1 relevant

    return {
        'mrr': mrr, 'map': map_score,
        **{f'recall@{k}': v for k, v in recall_at_k.items()},
        'mean_rank': ranks.mean(), 'median_rank': np.median(ranks),
    }

In [ ]:
# Train with early stopping on val loss
best_path = ARTIFACT_DIR / 'siamese_bilstm_best.pt'
best_val_loss = float('inf')
patience_counter = 0
history = []

print(f'{"Epoch":>5}  {"train_loss":>10}  {"val_loss":>8}  {"val_acc":>7}  {"pos_acc":>7}  {"neg_acc":>7}')
print('-' * 58)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val = evaluate_pairs(model, val_loader)

    history.append({
        'epoch': epoch, 'train_loss': train_loss,
        'val_loss': val['loss'], 'val_acc': val['acc'],
        'val_pos_acc': val['pos_acc'], 'val_neg_acc': val['neg_acc'],
    })

    print(f'{epoch:5d}  {train_loss:10.4f}  {val["loss"]:8.4f}  {val["acc"]:7.4f}  '
          f'{val["pos_acc"]:7.4f}  {val["neg_acc"]:7.4f}')

    if val['loss'] < best_val_loss:
        best_val_loss = val['loss']
        patience_counter = 0
        torch.save(model.state_dict(), best_path)
        print(f'  ✓ saved (val_loss={best_val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch}')
            break

In [ ]:
# Test set pair evaluation
model.load_state_dict(torch.load(best_path, map_location=device))
test = evaluate_pairs(model, test_loader)

print('--- Test Pair Results ---')
print(f'Loss:      {test["loss"]:.4f}')
print(f'Accuracy:  {test["acc"]:.4f}')
print(f'Pos Acc:   {test["pos_acc"]:.4f} (correctly identifying relevant)')
print(f'Neg Acc:   {test["neg_acc"]:.4f} (correctly identifying non-relevant)')

In [ ]:
# Full retrieval evaluation on test split
ret = retrieval_eval(model, test_q_ids, test_a_ids, corpus_test_ids)

print('--- Retrieval Results (test split) ---')
print(f'MRR:        {ret["mrr"]:.4f}')
print(f'mAP:        {ret["map"]:.4f}')
print(f'Recall@1:   {ret["recall@1"]:.4f}')
print(f'Recall@5:   {ret["recall@5"]:.4f}')
print(f'Recall@10:  {ret["recall@10"]:.4f}')
print(f'Recall@20:  {ret["recall@20"]:.4f}')
print(f'Mean rank:  {ret["mean_rank"]:.1f}  Median: {ret["median_rank"]:.1f}')

In [ ]:
# Save artifacts
metadata = {
    'model_name': 'SiameseBiLSTM',
    'reference': 'Neculoiu et al. 2016',
    'max_len': MAX_LEN,
    'num_bilstm_layers': NUM_BILSTM_LAYERS,
    'hidden_dim': HIDDEN_DIM,
    'dense_dim': DENSE_DIM,
    'dropout_recurrent': DROPOUT_RECURRENT,
    'dropout_inter': DROPOUT_INTER,
    'dropout_out': DROPOUT_OUT,
    'contrastive_margin': CONTRASTIVE_MARGIN,
    'negative_ratio': NEGATIVE_RATIO,
    'positive_loss_scale': POSITIVE_LOSS_SCALE,
    'batch_size': BATCH_SIZE,
    'adam_lr': ADAM_LR,
    'grad_clip': GRAD_CLIP,
    'vocab_size': len(stoi),
    'embed_dim': 300,
    'pad_token': PAD_TOKEN,
    'unk_token': UNK_TOKEN,
    'pad_idx': pad_idx,
    'best_val_loss': best_val_loss,
    'test_pair_acc': test['acc'],
    'test_retrieval': ret,
    'history': history,
}

with open(ARTIFACT_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# Also save encoder-only checkpoint for inference
torch.save(model.encoder.state_dict(), ARTIFACT_DIR / 'encoder.pt')
torch.save(stoi, ARTIFACT_DIR / 'stoi.pt')

print('Saved to', ARTIFACT_DIR)
print('  siamese_bilstm_best.pt  — full model')
print('  encoder.pt             — encoder only (inference)')
print('  stoi.pt                — vocabulary')
print('  metadata.json          — config + results')

In [ ]:
# Download artifacts (Colab)
from google.colab import files
import shutil

shutil.make_artifact(ARTIFACT_DIR.parent / 'siamese_bilstm_export.zip', ARTIFACT_DIR)
files.download(str(ARTIFACT_DIR.parent / 'siamese_bilstm_export.zip'))